In [20]:
import os
import csv
from bs4 import BeautifulSoup
from pathlib import Path
import pandas as pd

In [21]:
def extract_abstract(soup):
    """Extrae el abstract del HTML de PubMed Central"""
    abstract_text = ""
    
    # Buscar en diferentes posibles estructuras
    abstract_section = soup.find('div', class_='abstract') or \
                      soup.find('section', {'id': 'abstract'}) or \
                      soup.find('div', {'id': 'abstract'})
    
    if abstract_section:
        # Remover el título "Abstract" si existe
        for title in abstract_section.find_all(['h2', 'h3', 'title']):
            title.decompose()
        abstract_text = abstract_section.get_text(separator=' ', strip=True)
    
    return abstract_text

def extract_conclusions(soup):
    """Extrae las conclusiones del HTML de PubMed Central"""
    conclusions_text = ""
    
    # Buscar secciones de conclusiones con diferentes nombres posibles
    possible_headers = ['conclusion', 'conclusions', 'concluding remarks', 
                       'summary', 'conclusions and perspectives']
    
    for header in soup.find_all(['h2', 'h3', 'h4']):
        header_text = header.get_text().lower().strip()
        if any(term in header_text for term in possible_headers):
            # Obtener el contenido después del header
            conclusion_section = header.find_next_sibling()
            if conclusion_section:
                conclusions_text = conclusion_section.get_text(separator=' ', strip=True)
                break
    
    # También buscar por ID o clase
    if not conclusions_text:
        conclusion_div = soup.find('div', {'id': lambda x: x and 'conclusion' in x.lower()}) or \
                        soup.find('section', {'id': lambda x: x and 'conclusion' in x.lower()})
        if conclusion_div:
            conclusions_text = conclusion_div.get_text(separator=' ', strip=True)
    
    return conclusions_text

def process_pmc_htmls(folder_path='pmc_htmls', output_csv='pubmed_data.csv'):
    """Procesa todos los archivos HTML en la carpeta y crea el CSV"""
    
    folder = Path(folder_path)
    if not folder.exists():
        print(f"Error: La carpeta '{folder_path}' no existe")
        return
    
    results = []
    html_files = list(folder.glob('*.html'))
    
    if not html_files:
        print(f"No se encontraron archivos HTML en '{folder_path}'")
        return
    
    print(f"Procesando {len(html_files)} archivos...")
    
    for html_file in html_files:
        # Extraer el PMC ID del nombre del archivo (sin .html)
        pmc_id = html_file.stem
        
        try:
            with open(html_file, 'r', encoding='utf-8') as f:
                html_content = f.read()
            
            soup = BeautifulSoup(html_content, 'html.parser')
            
            abstract = extract_abstract(soup)
            conclusions = extract_conclusions(soup)
            
            results.append({
                'pmc_id': pmc_id,
                'abstract': abstract,
                'conclusions': conclusions
            })
            
            print(f"✓ Procesado: {pmc_id}")
            
        except Exception as e:
            print(f"✗ Error procesando {pmc_id}: {str(e)}")
            results.append({
                'pmc_id': pmc_id,
                'abstract': '',
                'conclusions': ''
            })
    
    # Escribir resultados al CSV
    with open(output_csv, 'w', newline='', encoding='utf-8') as csvfile:
        fieldnames = ['pmc_id', 'abstract', 'conclusions']
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        
        writer.writeheader()
        writer.writerows(results)
    
    print(f"\n✓ Archivo CSV creado: {output_csv}")
    print(f"Total de artículos procesados: {len(results)}")

if __name__ == "__main__":
    # Ejecutar el procesamiento
    process_pmc_htmls()
    
    # Si quieres personalizar las rutas:
    # process_pmc_htmls(folder_path='mi_carpeta', output_csv='mi_resultado.csv')

Procesando 572 archivos...
✓ Procesado: PMC11999716
✓ Procesado: PMC7076552
✓ Procesado: PMC4469364
✓ Procesado: PMC3251573
✓ Procesado: PMC10503492
✓ Procesado: PMC9953463
✓ Procesado: PMC5826609
✓ Procesado: PMC4917201
✓ Procesado: PMC9743659
✓ Procesado: PMC8754149
✓ Procesado: PMC10764921
✓ Procesado: PMC8396460
✓ Procesado: PMC9832585
✓ Procesado: PMC10285634
✓ Procesado: PMC10996920
✓ Procesado: PMC3166430
✓ Procesado: PMC4064004
✓ Procesado: PMC4321547
✓ Procesado: PMC10020673
✓ Procesado: PMC6379395
✓ Procesado: PMC7778922
✓ Procesado: PMC11470607
✓ Procesado: PMC3792163
✓ Procesado: PMC7561690
✓ Procesado: PMC3603133
✓ Procesado: PMC5460135
✓ Procesado: PMC11942576
✓ Procesado: PMC3869332
✓ Procesado: PMC11339457
✓ Procesado: PMC5568470
✓ Procesado: PMC11892206
✓ Procesado: PMC6036641
✓ Procesado: PMC3947616
✓ Procesado: PMC10848226
✓ Procesado: PMC8234954
✓ Procesado: PMC8185232
✓ Procesado: PMC7599661
✓ Procesado: PMC4085587
✓ Procesado: PMC4490751
✓ Procesado: PMC7118179
✓ 

In [26]:
df = pd.read_csv("pubmed_data.csv")
df.head()

,pmc_id,abstract,conclusions
0,PMC11999716,NaN,The microbiome of the Gowanus Canal is a biote...
1,PMC7076552,NaN,As the volume of spaceflight omics-level data ...
2,PMC4469364,NaN,NaN
3,PMC3251573,NaN,Plasmids Size (bp) Inc group GC% N° ORFs Start...
4,PMC10503492,NaN,Spaceflight poses risks to the central nervous...


In [23]:
dfr = pd.read_csv("articles_data_updated.csv")
dfr["Conclusions"] = df["conclusions"]
dfr.head()

,PMC_ID,Title,Authors,Introduction,Development/Methods,Results,Discussion,References_Count,References,Conclusions
0,PMC4136787,Mice in Bion-M 1 Space Mission: Training and S...,Alexander Andreev-Andrievskiy; Anfisa Popova; ...,"After a 16-year hiatus, Russia resumed in 2013...",The study was approved by IACUC of MSU Institu...,Living conditions for animals considered optim...,Living conditions for animals considered optim...,37,2007 Animals in space Vestnik Rossijskoj Akade...,The microbiome of the Gowanus Canal is a biote...
1,PMC3630201,Microgravity Induces Pelvic Bone Loss through ...,Elizabeth A. Blaber; Natalya Dvorochkin; Chial...,"On Earth, at 1 g, mechanical loading of mammal...",All experimental animal procedures for STS-131...,All flight and ground control mice were observ...,"In this study, we investigated cellular and mo...",74,2000 Historical overview of the Bion project J...,As the volume of spaceflight omics-level data ...
2,PMC11988870,Microgravity and Cellular Biology: Insights in...,Nelson Adolfo López Garzón; María Virginia Pin...,"Microgravity, a condition characterized by min...",A comprehensive literature review was conducte...,NaN,Recent research demonstrates that microgravity...,70,2003 Genetic models in applied physiology: sel...,NaN
3,PMC7998608,Selective Proliferation of Highly Functional A...,Takanobu Mashiko; Koji Kanayama; Natsumi Saito...,Human adipose-derived stem cells (hASCs) are e...,Human lipoaspirates were obtained from 12 heal...,Cells were expanded for three passages before ...,"Through novel advances in cell biology, adult ...",48,2013 Effects of spaceflight and ground recover...,Plasmids Size (bp) Inc group GC% N° ORFs Start...
4,PMC5587110,Microgravity validation of a novel system for ...,Macarena Parra; Jimmy Jung; Travis D. Boone; L...,The ISS National Laboratory is a unique resear...,"In order to validate the system, a number of g...",In order to assess the functionality of PCR in...,One of the major obstacles to space exploratio...,38,2013 Changes in Mouse Thymus and Spleen after ...,Spaceflight poses risks to the central nervous...


In [24]:
# Save the updated DataFrame
dfr.to_csv("articles_data_updated.csv", index=False, encoding='utf-8')

In [25]:
dfr.head()

,PMC_ID,Title,Authors,Introduction,Development/Methods,Results,Discussion,References_Count,References,Conclusions
0,PMC4136787,Mice in Bion-M 1 Space Mission: Training and S...,Alexander Andreev-Andrievskiy; Anfisa Popova; ...,"After a 16-year hiatus, Russia resumed in 2013...",The study was approved by IACUC of MSU Institu...,Living conditions for animals considered optim...,Living conditions for animals considered optim...,37,2007 Animals in space Vestnik Rossijskoj Akade...,The microbiome of the Gowanus Canal is a biote...
1,PMC3630201,Microgravity Induces Pelvic Bone Loss through ...,Elizabeth A. Blaber; Natalya Dvorochkin; Chial...,"On Earth, at 1 g, mechanical loading of mammal...",All experimental animal procedures for STS-131...,All flight and ground control mice were observ...,"In this study, we investigated cellular and mo...",74,2000 Historical overview of the Bion project J...,As the volume of spaceflight omics-level data ...
2,PMC11988870,Microgravity and Cellular Biology: Insights in...,Nelson Adolfo López Garzón; María Virginia Pin...,"Microgravity, a condition characterized by min...",A comprehensive literature review was conducte...,NaN,Recent research demonstrates that microgravity...,70,2003 Genetic models in applied physiology: sel...,NaN
3,PMC7998608,Selective Proliferation of Highly Functional A...,Takanobu Mashiko; Koji Kanayama; Natsumi Saito...,Human adipose-derived stem cells (hASCs) are e...,Human lipoaspirates were obtained from 12 heal...,Cells were expanded for three passages before ...,"Through novel advances in cell biology, adult ...",48,2013 Effects of spaceflight and ground recover...,Plasmids Size (bp) Inc group GC% N° ORFs Start...
4,PMC5587110,Microgravity validation of a novel system for ...,Macarena Parra; Jimmy Jung; Travis D. Boone; L...,The ISS National Laboratory is a unique resear...,"In order to validate the system, a number of g...",In order to assess the functionality of PCR in...,One of the major obstacles to space exploratio...,38,2013 Changes in Mouse Thymus and Spleen after ...,Spaceflight poses risks to the central nervous...
